# Transcription Factor Activity in Differentiated and Stem-like Malignant States

Infers per-cell TF activity with decoupler (univariate linear model, ULM) using two
prior regulon networks, CollecTRI and DoRothEA in the
differentiated epithelial and developmental stem-like malignant states. TFs are
ranked per state × treatment-stage group against all other groups, summarised as
heatmaps, and a curated TF panel is tested across treatment stages at patient level.

In [ ]:
# !pip install --quiet scanpy decoupler statsmodels
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/tumour/pulmonary_epi_Lin/

In [ ]:
import os

import decoupler as dc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

## Config

In [ ]:
input_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states_join_cytotrace_monocle_malignant.h5ad"
figures_output_dir = "./figures/tf_activity"
tables_output_dir = "./tables"

state_col = "final_state"
recist_col = "recist_v3"
donor_col = "donor"
counts_layer = "raw_counts"
group_col = "state_recist"
min_cells_per_gene = 10

states = ["2_Differentiated epithelial", "3_Devp_stem-like"]
treatment_order = ["1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]

# --- TF activity (decoupler ULM) ---
networks = ["CollecTRI", "DoRothEA"]
dorothea_levels = ["A", "B", "C"]
ulm_tmin = 5                     # minimum targets per TF after matching to measured genes

# --- TF ranking: each group vs the other state x stage groups ---
rank_method = "wilcoxon"
padj_cutoff = 0.05
n_markers = 20                   # top TFs per group in the overview heatmap
marker_selection = "up"          # "up": stat > 0, ranked by stat; "original": decoupler padj order (mixes up- and down-regulated)

# --- curated TF panel ---
tf_panel = ["TEAD1", "LGR4", "KLF3", "KLF6", "TCF7", "ATF7", "KMT2A", "TCF7L2", "AXIN2", "LEF1", "SOX5"]
tf_panel_network = "DoRothEA"
panel_mark_z = 0.75              # "*" where z-scaled mean activity > this; None to disable (display threshold only)

# --- patient-level stage comparison for the panel ---
min_cells_per_patient = 20
correction_method = "fdr_bh"
alpha = 0.05

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)

## 1. Load, subset and normalise

`group_col` combines state and treatment stage; groups are ordered by state, then stage.

In [ ]:
adata = sc.read(input_path)
adata = adata[adata.obs[state_col].isin(states) & adata.obs[recist_col].isin(treatment_order)].copy()

adata.X = adata.layers[counts_layer].copy()
adata.uns.pop("log1p", None)
sc.pp.filter_genes(adata, min_cells=min_cells_per_gene)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

labels = adata.obs[state_col].astype(str) + " | " + adata.obs[recist_col].astype(str)
group_order = [f"{s} | {t}" for s in states for t in treatment_order if f"{s} | {t}" in set(labels)]
adata.obs[group_col] = pd.Categorical(labels, categories=group_order, ordered=True)

adata.obs[group_col].value_counts(sort=False)

## 2. Infer TF activity

In [ ]:
def load_network(name):
    if name == "CollecTRI":
        return dc.op.collectri(organism="human")
    if name == "DoRothEA":
        net = dc.op.dorothea(organism="human")
        return net[net["confidence"].isin(dorothea_levels)]
    raise ValueError(name)

tf_scores = {}
for name in networks:
    ad = adata.copy()
    dc.mt.ulm(data=ad, net=load_network(name), tmin=ulm_tmin)
    score = dc.pp.get_obsm(ad, key="score_ulm")
    score.obs = ad.obs.copy()
    tf_scores[name] = score
    print(f"{name}: {score.n_vars} TFs scored")

## 3. Rank TFs per group

Wilcoxon test of each group against all other groups in this subset. Cell-level
tests at this n give padj ≈ 0 for most TFs, so ranking uses the test statistic;
see section 6 for a patient-level check.

In [ ]:
def select_markers(ranked, n):
    sig = ranked[ranked["padj"] < padj_cutoff]
    if marker_selection == "up":
        sig = sig[sig["stat"] > 0].sort_values(["group", "stat"], ascending=[True, False])
    top = sig.groupby("group", observed=True).head(n)
    return list(dict.fromkeys(top["name"].astype(str)))

tf_ranked, tf_markers = {}, {}
for name, score in tf_scores.items():
    ranked = dc.tl.rankby_group(adata=score, groupby=group_col, reference="rest", method=rank_method)
    ranked.to_csv(f"{tables_output_dir}/{name.lower()}_ranked_tfs.csv", index=False)
    tf_ranked[name] = ranked
    tf_markers[name] = select_markers(ranked, n_markers)
    print(f"{name}: {len(tf_markers[name])} marker TFs")

## 4. Top TFs per group

Colour: mean ULM score per group, z-scaled across groups for each TF. Circle size:
fraction of cells with ULM score > 0 (positive inferred activity).

In [ ]:
def group_summary(score, tfs):
    X = score[:, tfs].to_df()
    groups = score.obs[group_col]
    mean = X.groupby(groups, observed=True).mean().loc[group_order]
    frac = (X > 0).groupby(groups, observed=True).mean().loc[group_order]
    z = ((mean - mean.mean()) / mean.std()).fillna(0)
    return z, frac

def plot_tf_overview(score, tfs, title, save_path=None):
    z, frac = group_summary(score, tfs)
    fig, ax = plt.subplots(figsize=(max(14, 0.45 * len(tfs)), max(7, 0.65 * len(group_order))))
    sns.heatmap(z, cmap="coolwarm", center=0, linewidths=0.4, linecolor="lightgrey",
                cbar_kws={"label": "Z-scaled TF activity"}, ax=ax)
    yy, xx = np.meshgrid(np.arange(z.shape[0]), np.arange(z.shape[1]), indexing="ij")
    ax.scatter(xx.ravel() + 0.5, yy.ravel() + 0.5, s=10 + 100 * frac.values.ravel(),
               facecolors="none", edgecolors="black", linewidth=1)

    state_of = [g.split(" | ")[0] for g in group_order]
    for y in range(1, len(group_order)):
        if state_of[y] != state_of[y - 1]:
            ax.axhline(y, color="black", linewidth=1.2)

    handles = [ax.scatter([], [], s=10 + 100 * v, facecolors="none", edgecolors="black") for v in (0.25, 0.5, 0.75, 1)]
    ax.legend(handles, ["25%", "50%", "75%", "100%"], title="Fraction score > 0",
              frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.set_title(title, fontsize=14, weight="bold")
    ax.set(xlabel="Transcription factors", ylabel="")
    ax.tick_params(axis="x", labelsize=9, rotation=90)
    ax.tick_params(axis="y", labelsize=10, rotation=0)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

for name, score in tf_scores.items():
    if not tf_markers[name]:
        print(f"{name}: no TFs pass padj < {padj_cutoff}; skipped")
        continue
    plot_tf_overview(score, tf_markers[name], f"{name} TF activity",
                     save_path=f"{figures_output_dir}/{name.lower()}_tf_heatmap.pdf")

## 5. Curated TF panel

TFs as rows (clustered), groups as columns. `*` marks z > `panel_mark_z`; it is a
display threshold, not a significance test. Panel entries without a regulon in the
network are reported and skipped.

In [ ]:
score = tf_scores[tf_panel_network]
panel_use = [g for g in tf_panel if g in score.var_names]
panel_missing = [g for g in tf_panel if g not in score.var_names]
if panel_missing:
    print(f"no {tf_panel_network} regulon (skipped): {panel_missing}")

z, _ = group_summary(score, panel_use)
plot_mat = z.T
if plot_mat.shape[0] > 2:
    cg = sns.clustermap(plot_mat, row_cluster=True, col_cluster=False)
    plot_mat = plot_mat.iloc[cg.dendrogram_row.reordered_ind]
    plt.close(cg.fig)

fig, ax = plt.subplots(figsize=(2.5, 3))
sns.heatmap(plot_mat, cmap="vlag", center=0, linewidths=0.4, linecolor="lightgrey",
            cbar_kws={"label": "Z-scaled TF activity", "shrink": 0.8}, ax=ax)
if panel_mark_z is not None:
    for i, j in zip(*np.where(plot_mat.values > panel_mark_z)):
        ax.text(j + 0.5, i + 0.5, "*", ha="center", va="center", fontsize=14, color="white", fontweight="bold")
ax.set_title(f"{tf_panel_network} regulatory transition", fontsize=13, weight="bold", pad=10)
ax.set(xlabel="", ylabel="")
ax.tick_params(axis="x", labelsize=10, rotation=90)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10, style="italic")
plt.savefig(f"{figures_output_dir}/{tf_panel_network.lower()}_tf_panel_heatmap.pdf", dpi=300, bbox_inches="tight")
plt.show()

## 6. Panel TFs across treatment stages (patient level)

Per state and TF: median ULM score per patient, Kruskal–Wallis across stages, then
pairwise Mann–Whitney where the omnibus test is significant (corrected within
state × TF). The cell-level `rankby_group` results for the panel are saved alongside.

In [ ]:
obs = score.obs[[state_col, recist_col, donor_col]].astype(str)
panel_df = pd.concat([obs, score[:, panel_use].to_df()], axis=1)
grouped = panel_df.groupby([state_col, recist_col, donor_col])
per_patient = grouped[panel_use].median().join(grouped.size().rename("n_cells")).reset_index()
per_patient = per_patient[per_patient["n_cells"] >= min_cells_per_patient]

rows = []
for state in states:
    for tf in panel_use:
        d = per_patient[per_patient[state_col] == state]
        groups = {t: d.loc[d[recist_col] == t, tf].values for t in treatment_order}
        groups = {t: v for t, v in groups.items() if len(v) >= 2}
        n = {t: len(v) for t, v in groups.items()}
        if len(groups) < 2:
            continue
        h, p_omni = kruskal(*groups.values())
        rows.append({"state": state, "tf": tf, "test": "kruskal", "comparison": "all", "p": p_omni, "n": str(n)})
        if p_omni >= alpha:
            continue
        stages = list(groups)
        posthoc = []
        for i in range(len(stages)):
            for j in range(i + 1, len(stages)):
                a, b = stages[i], stages[j]
                _, p = mannwhitneyu(groups[a], groups[b], alternative="two-sided")
                posthoc.append({"state": state, "tf": tf, "test": "mannwhitneyu", "comparison": f"{a} vs {b}",
                                "p": p, "n": f"{n[a]} vs {n[b]}",
                                "median_a": np.median(groups[a]), "median_b": np.median(groups[b])})
        for r, pa in zip(posthoc, multipletests([r["p"] for r in posthoc], method=correction_method)[1]):
            r["p_adj"] = pa
        rows.extend(posthoc)

panel_stats = pd.DataFrame(rows)
panel_stats.to_csv(f"{tables_output_dir}/{tf_panel_network.lower()}_tf_panel_stage_stats_patient.csv", index=False)

ranked = tf_ranked[tf_panel_network]
ranked[ranked["name"].isin(panel_use)].sort_values(["name", "group"]).to_csv(
    f"{tables_output_dir}/{tf_panel_network.lower()}_tf_panel_ranked_cell_level.csv", index=False)
panel_stats